In [1]:
import scanpy as sc
import numpy as np
import pandas as pd
import torch
import umap
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
import seaborn as sns
from models.deep_multi_encoder import MultiEncoderAutoencoder 

/scratch/work/sagara22/newXencoder/.newEnv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
adata1 = sc.read_h5ad("sq_cell_feature_1.h5ad")
adata2 = sc.read_h5ad("sq_cell_feature_2.h5ad")

In [3]:
adata1, adata2

(AnnData object with n_obs × n_vars = 245754 × 289
     obs: 'cell_id', 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'pct_counts_in_top_10_genes', 'pct_counts_in_top_20_genes', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_150_genes', 'n_counts', 'leiden'
     var: 'gene_ids', 'feature_types', 'genome', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells'
     uns: 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'umap'
     obsm: 'X_pca', 'X_umap', 'spatial'
     varm: 'PCs'
     obsp: 'connectivities', 'distances',
 AnnData object with n_obs × n_vars = 271693 × 5001
     obs: 'cell_id', 'x_centroid', 'y_cent

In [4]:
X1 = adata1.X.toarray()
X2 = adata2.X.toarray()

In [5]:
X1_tensor = torch.tensor(X1, dtype=torch.float32)
X2_tensor = torch.tensor(X2, dtype=torch.float32)

In [6]:
adata1.shape[1]

289

In [7]:
# latent_dim = 384
# hidden_dim = 128
# model = MultiEncoderAutoencoder(source_dim=289, target_dim=5001, latent_dim=latent_dim, hidden_dim=hidden_dim)

# # Load trained weights
# checkpoint = torch.load("/scratch/work/sagara22/newXencoder/new/models/best_model_20250328_021813.pt", map_location="cpu")
# model.load_state_dict(checkpoint)
# model.to("cuda")
# model.eval()
# X1_tensor = X1_tensor.to("cuda")
# X2_tensor = X2_tensor.to("cuda")

In [9]:
latent_dim = 20
hidden_dim = 2
model = MultiEncoderAutoencoder(source_dim=289, target_dim=5001, latent_dim=latent_dim, hidden_dim=hidden_dim)

# Load trained weights
checkpoint = torch.load("/scratch/work/sagara22/newXencoder/deepMMD_L_512/models/best_model_20250404_122523.pt")
model.load_state_dict(checkpoint)
model.to("cuda")
model.eval()
X1_tensor = X1_tensor.to("cuda")
X2_tensor = X2_tensor.to("cuda")

In [ ]:
with torch.no_grad():
    z1 = model.encode_source(X1_tensor).cpu().numpy()
    z2 = model.encode_target(X2_tensor).cpu().numpy()

# Apply UMAP separately to each latent space
umap_model1 = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, metric='euclidean')
Z1_umap = umap_model1.fit_transform(z1)

umap_model2 = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, metric='euclidean')
Z2_umap = umap_model2.fit_transform(z2)

In [ ]:
leiden1 = adata1.obs["leiden"].astype(str).values
leiden2 = adata2.obs["leiden"].astype(str).values

In [ ]:
cl_annotations1 = {
    '1': "Epithelial", '2': "Fibroblasts", '0': "T cells", '3': "Macrophages", '4': "Endothelial",
    '5': "Endothelial", '6': "Smooth muscles", '7': "Epithelial", '8': "Epithelial", '9': "B cells",
    '10': "Epithelial", '11': "Plasma cells", '12': "Epithelial"
}

cl_annotations2 = {
    '1': "Endothelial", '2': "Fibroblasts", '0': "T cells", '4': "Macrophages", '3': "Epithelial",
    '5': "Endothelial", '6': "Epithelial", '7': "Epithelial", '8': "B cells", '9': "Endothelial",
    '10': "Mast cells", '11': "Macrophages"
}


In [ ]:
# Annotate clusters with both Leiden and cell type
# Shorten cell type labels: "Epithelial [1]" instead of "1 (Epithelial)"
cell_types1 = [f"{cl_annotations1.get(leiden, 'Unknown')} [{leiden}]" for leiden in leiden1]
cell_types2 = [f"{cl_annotations2.get(leiden, 'Unknown')} [{leiden}]" for leiden in leiden2]

In [ ]:
# Plot UMAP for dataset 1
plt.figure(figsize=(8, 6))
sns.scatterplot(
    x=Z1_umap[:, 0], y=Z1_umap[:, 1], 
    hue=cell_types1, palette="tab20", s=5, alpha=0.8, edgecolor=None
)
plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.title("Latent Space UMAP - Dataset 1")
plt.legend(title="Leiden (Cell Type)", bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig('deepMMD_L_384/latent_umap_1.png', dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(
    x=Z2_umap[:, 0], y=Z2_umap[:, 1], 
    hue=cell_types2, palette="tab20", s=5, alpha=0.8, edgecolor=None
)
plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.title("Latent Space UMAP - Dataset 2")
plt.legend(title="Leiden (Cell Type)", bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig('deepMMD_L_384/latent_umap_2.png', dpi=300)
plt.show()